# 1. Base Setup

## 1.1 Install packages

In [3]:
# %load_ext autoreload
# %autoreload 2

In [4]:
# !pip install kagglehub
# !pip install statsmodels
# !pip install xgboost
# !pip install catboost

## 1.2 Load all needed imports

In [5]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint, uniform
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [6]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd()
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [7]:
def data_cleaning(df: pd.DataFrame, predict: bool=False) -> tuple:
    """Clean raw data and split into model and demo DataFrames.

    Deduplicates rows, parses dates, renames misspelled columns, selects
    relevant columns, and splits into a model_df (paid invoices only)
    and demo_df (all invoices).

    Args:
        df: raw DataFrame from load_cashflow_data().

    Returns:
        A tuple (model_df, demo_df).
    """
    # to remove warning of SettingWithCopyWarning:
    df = df.copy()

    # drop duplicates and name stripping
    df = df.drop_duplicates()
    df.columns = df.columns.str.strip()

    if "cust_number" in df.columns:
        df["cust_number"] = df["cust_number"].astype(str)
    # Drop invalid invoice ids
    df = df.dropna(subset=["invoice_id"]).copy()
    df["invoice_id"] = pd.to_numeric(df["invoice_id"], errors="coerce")
    df = df[df["invoice_id"].notna()].copy()

    # Parse date columns
    df["due_in_date"] = pd.to_datetime(df["due_in_date"], format="%Y%m%d", errors="coerce")
    df["baseline_create_date"] = pd.to_datetime(df["baseline_create_date"], format="%Y%m%d", errors="coerce")
    df["clear_date"] = pd.to_datetime(df["clear_date"], errors="coerce")
    # Cast IDs
    df["doc_id"] = df["doc_id"].astype("int64")

    # Standardize categorical columns
    cat_cols = ["business_code", "name_customer", "invoice_currency", "document type", "cust_payment_terms"]
    for c in cat_cols:
        df[c] = df[c].astype(str)

    # Select and rename columns
    df = df[['doc_id','cust_number', 'buisness_year', 'due_in_date', 'invoice_currency',
             'document type', 'total_open_amount', 'baseline_create_date',
             'cust_payment_terms', 'clear_date']]

    df.rename(columns={
        'buisness_year': 'business_year',
        'clear_date': 'invoice_paid',
        'document type': 'document_type',
        'baseline_create_date': 'invoice_sent',
    }, inplace=True)

    # sort values
    df = df.sort_values("invoice_sent").reset_index(drop=True)
    # data conversion
    df["business_year"] = df["business_year"].round().astype("int64")

    # converting cad to usd
    cad_to_usd_by_year = {
        2018 : 0.771,
        2019 : 0.754,
        2020 : 0.745
    }
    def convert_cad_to_usd_amount(row):
        if row['invoice_currency'].strip().upper() == "USD":
            return row["total_open_amount"]
        elif row['invoice_currency'].strip().upper() == "CAD":
            year = int(row['business_year'])
            rate = cad_to_usd_by_year.get(year,0.75)
            return row["total_open_amount"] * rate
        else:
            return row["total_open_amount"]

    df["total_open_amount"] = df.apply(convert_cad_to_usd_amount,axis=1)

    if not predict:
        df = df[df["invoice_paid"].notnull()]
    # Save processed frames
    base_dir = Path.cwd()
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    # if not predict:
        # df.to_csv(raw_data_dir / "df.csv", index=False)
    #demo_df.to_csv(raw_data_dir / "demo_df.csv", index=False)

    #print(f"Saved df ({len(df)} rows) and demo_df ({len(demo_df)} rows)")
    return df

In [8]:
def engineer_features(snapshot: pd.DataFrame, df_full: pd.DataFrame,
                      current_date) -> pd.DataFrame:
    """Engineer all features for a given reference date.

    Adds invoice timing features, customer behaviour features (from full
    history), and amount-based features to the snapshot.

    Args:
        snapshot: DataFrame of open invoices at current_date.
        df_full: full invoice DataFrame (for historical calculations).
        current_date: the reference date for the snapshot.

    Returns:
        The snapshot DataFrame with all engineered features added.
    """
    # A) Invoice timing features
    open_invoice = snapshot["invoice_paid"].isna() | (snapshot["invoice_paid"] > current_date)
    snapshot["invoice_age_days"] = np.where(
        open_invoice,
        (current_date - snapshot["invoice_sent"]).dt.days,
        np.nan,
    )
    snapshot["days_until_due"] = np.where(
        open_invoice,
        (snapshot["due_in_date"] - current_date).dt.days,
        np.nan,
    )
    snapshot["pay_terms_days"] = (snapshot["due_in_date"] - snapshot["invoice_sent"]).dt.days
    snapshot["invoice_month"] = snapshot["invoice_sent"].dt.month
    snapshot["due_month"] = snapshot["due_in_date"].dt.month
    snapshot["days_past_due"] = (current_date - snapshot["due_in_date"]).dt.days

    # B) Customer behaviour features
    historical = df_full[df_full["invoice_paid"] <= current_date].copy()
    historical["delay"] = (historical["invoice_paid"] - historical["due_in_date"]).dt.days.clip(lower=0)
    avg_delay = historical.groupby("cust_number")["delay"].mean().rename("customer_avg_delay")

    historical["is_late"] = (historical["invoice_paid"] > historical["due_in_date"]).astype(int)
    late_ratio = historical.groupby("cust_number")["is_late"].mean().rename("late_payment_ratio")

    before_current = df_full[df_full["invoice_sent"] < current_date]
    prev_counts = before_current.groupby("cust_number").size().rename("prev_transaction_count")

    last_invoice = (
        before_current.groupby("cust_number")["invoice_sent"]
        .max()
        .rename("last_invoice_date")
    )

    customer_features = pd.concat([avg_delay, late_ratio, prev_counts, last_invoice], axis=1)
    snapshot = snapshot.merge(customer_features, on="cust_number", how="left")

    snapshot["customer_risk_score"] = (
        0.7 * snapshot["late_payment_ratio"].fillna(0)
        + 0.3 * snapshot["customer_avg_delay"].fillna(0)
    )

    snapshot["days_since_last_invoice"] = (current_date - snapshot["last_invoice_date"]).dt.days
    snapshot.drop(columns=["last_invoice_date"], inplace=True)
    snapshot["prev_transaction_count"] = snapshot["prev_transaction_count"].fillna(0).astype(int)

    # C) Invoice characteristics
    snapshot["invoice_amount"] = snapshot["total_open_amount"]
    snapshot["invoice_amount_log"] = np.log1p(snapshot["invoice_amount"].clip(lower=0))
    snapshot["open_amount"] = snapshot["total_open_amount"]

    size_bins_fixed = [-np.inf, 10000, 100000, np.inf]
    size_labels = ["small", "medium", "large"]
    snapshot["invoice_size_cat"] = pd.cut(
        snapshot["invoice_amount"], bins=size_bins_fixed, labels=size_labels,
    )

    q1, q2 = df_full["total_open_amount"].quantile([0.33, 0.66])
    size_bins_quant = [-np.inf, q1, q2, np.inf]
    snapshot["invoice_size_cat_q"] = pd.cut(
        snapshot["invoice_amount"], bins=size_bins_quant, labels=size_labels,
    )

    return snapshot

In [9]:
def build_sliding_window_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    """Build augmented dataset by sliding a weekly window over invoice history.

    For each weekly snapshot, selects all invoices sent but not yet paid,
    engineers features, and computes the target 'week_bucket' (1-7).

    Args:
        df: cleaned DataFrame with 'invoice_sent' and 'invoice_paid' columns.

    Returns:
        Concatenated DataFrame of all weekly snapshots with target column.
    """
    all_snapshots = []

    start_date = df["invoice_sent"].min()
    end_date = df["invoice_paid"].max() - pd.Timedelta(weeks=6)
    stride = pd.Timedelta(weeks=1)

    current = start_date
    while current <= end_date:
        snapshot = df[
            (df["invoice_sent"] <= current) &
            (df["invoice_paid"] > current)
        ].copy()

        snapshot = engineer_features(snapshot, df, current)

        snapshot["reference_date"] = current
        snapshot["days_to_payment"] = (snapshot["invoice_paid"] - current).dt.days
        snapshot["week_bucket"] = np.ceil(snapshot["days_to_payment"] / 7).astype(int)
        snapshot["week_bucket"] = snapshot["week_bucket"].clip(lower=1)
        snapshot.loc[snapshot["week_bucket"] > 7, "week_bucket"] = 7

        all_snapshots.append(snapshot)
        current += stride

    big_df = pd.concat(all_snapshots, ignore_index=True)
    print(f"Original rows: {len(df)}")
    print(f"Augmented rows: {len(big_df)}")
    print(big_df["week_bucket"].value_counts().sort_index())

    return big_df

In [10]:
NUMERIC_FEATURES = [
    "business_year", "invoice_age_days", "days_until_due", "pay_terms_days",
    "invoice_month", "due_month", "days_past_due", "customer_avg_delay",
    "late_payment_ratio", "prev_transaction_count", "days_since_last_invoice",
    "customer_risk_score", "invoice_amount", "invoice_amount_log",
]

CATEGORICAL_FEATURES = [
    "invoice_currency", "document_type", "invoice_size_cat", "invoice_size_cat_q"
]


def preprocess(df: pd.DataFrame, inference: bool = False) -> tuple:
    """Preprocess a DataFrame for model training or inference.

    Imputes missing values in customer history features and splits into
    feature matrix and (optionally) target vector. Identifier columns,
    date columns, and leaky columns are excluded.

    Args:
        df: pandas DataFrame containing invoice-level records.
        inference: if True, skips dropping rows without a target and
                   returns y as None. Set this when predicting on new
                   invoices that have no 'week_bucket' column yet.

    Returns:
        A tuple (X, y) where X is a DataFrame of feature columns and y is
        a Series of 'week_bucket' target labels (or None during inference).
    """
    df = df.copy()

    if not inference:
        df = df.dropna(subset=["week_bucket"])

    df["customer_avg_delay"] = df["customer_avg_delay"].fillna(0)
    df["late_payment_ratio"] = df["late_payment_ratio"].fillna(0)
    df["days_since_last_invoice"] = df["days_since_last_invoice"].fillna(-1)

    drop_cols = [
        "doc_id","cust_number", "due_in_date", "invoice_sent", "total_open_amount",
        "open_amount", "reference_date", "week_bucket", "days_to_payment",
        "invoice_paid", "cust_payment_terms"
    ]
    feature_cols = [c for c in df.columns if c not in drop_cols]

    X = df[feature_cols]
    y = df["week_bucket"] if "week_bucket" in df.columns and not inference else None

    return X, y

In [11]:
hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)
cutoff_date = big_df["invoice_sent"].quantile(0.8)

train_df = big_df[big_df["reference_date"] <= cutoff_date]
test_df = big_df[big_df["reference_date"] > cutoff_date]

X_train, y_train = preprocess(train_df)
X_test, y_test = preprocess(test_df)

Loading local file: /Users/anton/code/DERNTOAN/cf_copilot/notebooks/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64


In [12]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [23]:
from xgboost import XGBClassifier
from sklearn.feature_selection import VarianceThreshold

classifier = XGBClassifier(tree_method="hist")

XGB_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("variance", VarianceThreshold(threshold=0.05)),
    ("classifier", classifier),
])

In [24]:
cutoff_date = big_df["reference_date"].quantile(0.8)
test_fold = np.where(big_df["reference_date"] <= cutoff_date, -1, 0)
ps = PredefinedSplit(test_fold)

X, y = preprocess(big_df)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder

# This is XGBoost specific
le = LabelEncoder()
y_enc = le.fit_transform(y)

param_grids = {
    "RandomForest": {
        "classifier__n_estimators": [300, 350, 400, 450, 500],
        "classifier__max_depth": [16, 18, 20, 22, 25, None],
        "classifier__min_samples_split": [6, 8, 10, 12, 14],
        "classifier__min_samples_leaf": [2, 3, 4, 5, 7],
        "classifier__max_features": [0.2, 0.25, 0.3, 0.35, 0.4],
        "classifier__class_weight": ["balanced", "balanced_subsample"],
    },
    "XGBoost": {
        "classifier__n_estimators": randint(200, 350),
        "classifier__max_depth": randint(10, 15),
        "classifier__learning_rate": uniform(0.01, 0.04),       # 0.01 to 0.05
        "classifier__subsample": uniform(0.9, 0.1),             # 0.9 to 1.0
        "classifier__colsample_bytree": uniform(0.45, 0.2),     # 0.45 to 0.65
        "classifier__reg_alpha": uniform(0.4, 0.6),             # 0.4 to 1.0
        "classifier__reg_lambda": uniform(5.0, 4.0),            # 5.0 to 9.0
        "classifier__min_child_weight": randint(10, 18),
        "classifier__gamma": uniform(0.1, 0.2),                 # 0.1 to 0.3
    },
    "GradientBoosting": {
        "classifier__n_estimators": [100, 300, 500, 800],
        "classifier__max_depth": [3, 4, 5, 6, 8],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "classifier__min_samples_split": [2, 5, 10, 20],
        "classifier__min_samples_leaf": [1, 5, 10, 20],
        "classifier__max_features": ["sqrt", "log2", 0.3, 0.5],
    },
}

model_name = "XGBoost"

tscv = TimeSeriesSplit(n_splits=5)

xgb_random_search = RandomizedSearchCV(
    XGB_pipeline,
    param_distributions=param_grids[model_name],
    n_iter=50,
    cv=tscv,
    scoring="neg_log_loss",
    n_jobs=-1,
    verbose=1,
    random_state=42,
)

xgb_random_search.fit(X, y_enc)
print(f"Best log_loss: {-xgb_random_search.best_score_:.4f}")
print(f"Best params: {xgb_random_search.best_params_}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits


In [17]:
def evaluate_model(probas, preds):
    return log_loss(y_test, probas)

In [26]:
from sklearn.metrics import f1_score, accuracy_score
from cf_copilot.cashflow_prediction.evaluation import evaluate_forecast_holdout

y_test_enc = le.fit_transform(y_test)
y_train_enc = le.fit_transform(y_train)

searches = {
    "XGBoost": xgb_random_search
}

results = []

for name, search in searches.items():
    print(f"Evaluating {name}...")
    try:
        best_pipe = search.best_estimator_
        best_pipe.fit(X_train, y_train_enc)

        probas = best_pipe.predict_proba(X_test)
        preds = best_pipe.predict(X_test)
        forecast, _ = evaluate_forecast_holdout(best_pipe, test_df)

        result = {
            "model": name,
            "best_cv_score": search.best_score_,
            "log_loss": evaluate_model(probas, preds),
            "f1_macro": f1_score(y_test_enc, preds, average="macro"),
            "accuracy": accuracy_score(y_test_enc, preds),
            "best_params": str(search.best_params_),
        }
        for key, value in forecast.items():
            result[key] = value

        results.append(result)
    except Exception as e:
        print(f"  {name} failed: {e}")

results_df = pd.DataFrame(results).sort_values("log_loss")

Evaluating XGBoost...

Evaluating forecast on 21811 rows...
✅ Forecast MAE weekly: 2630336.80
✅ Forecast MAPE weekly: 0.8300
✅ Forecast total actual cf: 30433551.42
✅ Forecast total forecast cf: 17213350.21
✅ Forecast total cf difference: -13220201.21
